# Using `dzack_research.preamble`

The preamble supplies mathematical objects and morphisms for interactive Sage research.
This notebook checks representative constructions from their defining properties.
Run it from top to bottom in the SageMath kernel.

## Session helpers loaded by `init.sage`

In [ ]:
from dzack_research.preamble.install import install_preamble

install_preamble(globals())
Lattices.install(globals())

## Named lattices and constructors for $L_{\mathrm{K3}}$

In [ ]:
for lattice in (Lattices.U, Lattices.E8, Lattices.E10):
    show(lattice)

In [ ]:
Lattices.LK3

In [ ]:
A2.<alpha1, alpha2> = Lattices("A", 2)

assert alpha1 * alpha1 == 2
assert alpha1 * alpha2 == -1
assert alpha1.div() == 1
A2

In [ ]:
L = Lattices.TEn
L

In [13]:
L.discriminant_group()

QuadraticFormMorphism on Finitely presented module on 12 generators over Integer Ring with values in Q/2Z

## Predicates and isotropic quotients

In [14]:
L.discriminant_group().normal_form()

AttributeError: 'BasedFreeModule_with_category' object has no attribute 'submodule'

In [ ]:
e, f = Lattices.U.gens()

assert Lattices.E8.is_elliptic()
assert Lattices.U.q(e) == 0
assert e * f == 1

{
    "E8 is elliptic": Lattices.E8.is_elliptic(),
    "delta(E8)": Lattices.E8.delta(),
    "q(e)": Lattices.U.q(e),
    "e_perp / <e>": e.e_perp_mod_e(),
}

## Free algebra constructions on a presented module

In [ ]:
F = BasedFreeModule(ZZ, Sets.Δ[0])
x = F.module_generator(0)
M = FinitelyPresentedModule(module_homset(F, F)({0: 2 * x}))

T = TensorAlgebraOf(M)
Γ = DividedPowerAlgebraOf(M)
t = T.algebra_generator(0)
γ = Γ.algebra_generator(0)

assert 2 * t == T.zero()
assert t * t != T.zero()
assert 2 * (t * t) == T.zero()
assert Γ.graded_piece(2).invariants() == (4,)
assert 4 * Γ.divided_power(γ, 2) == Γ.zero()

T, Γ

## A nondegenerate form identifies a module with its dual

In [ ]:
gram = Lattices.U.gram_tensor()
identity = Lattices.U.raise_index(gram, 0)
correlation = Lattices.U.correlation_isomorphism()

assert identity.valence() == (1, 1)
assert identity.components() == {(0, 0): 1, (1, 1): 1}
assert Lattices.U.lower_index(identity, 0) == gram
assert all(
    correlation.inverse()(correlation(v)) == v
    for v in Lattices.U.module_generators()
)
identity

## Sterk root configurations

In [ ]:
sterk_configurations = Sterk.sterk_roots()
{
    name: {
        "number of roots": len(roots),
        "Gram rank": Lattices.TdP.gram_of(roots).rank(),
    }
    for name, roots in sterk_configurations.items()
}

In [ ]:
isotropic = Sterk.isotropic_vectors()["s4_12"]
Lattices.TdP.b(isotropic, isotropic)

## Coxeter diagrams as Sage parents and their morphisms

In [ ]:
A3_diagram = FiniteCoxeterDiagram.from_cartan_type(["A", 3])
A4_diagram = FiniteCoxeterDiagram.from_cartan_type(["A", 4])
inclusion = A3_diagram.hom([2, 3, 4], codomain=A4_diagram)

{
    "category": A3_diagram.category(),
    "in CoxeterDiagrams": A3_diagram in CoxeterDiagrams(),
    "Coxeter matrix": A3_diagram.coxeter_matrix(),
    "labeled edges": list(A3_diagram.graph().edges(sort=True)),
    "images": inclusion.images(),
}

In [ ]:
all(
    A3_diagram.coxeter_matrix()[s, t]
    == A4_diagram.coxeter_matrix()[
        inclusion(A3_diagram(s)).value,
        inclusion(A3_diagram(t)).value,
    ]
    for s in A3_diagram.index_set()
    for t in A3_diagram.index_set()
)

## Mixed tensors form one bigraded algebra

In [ ]:
from dzack_research.preamble.categories.modules.tensors import MixedTensorAlgebra

M2 = BasedFreeModule(ZZ, Sets.Δ[1])
A = MixedTensorAlgebra(M2)
V = A.homogeneous_piece((1, 0))
Vdual = A.homogeneous_piece((0, 1))
v = A.include(V({(0,): 1}))
φ = A.include(Vdual({(0,): 2, (1,): 3}))

assert A.dual_module() is DualModule(M2)
assert (v * φ).valences() == ((1, 1),)
assert A.one() * v == v
v * φ

## The K3 lattice as a named mathematical object

In [ ]:
LK3 = Lattices.LK3

assert LK3.rank() == 22
assert LK3.is_even()
assert LK3.is_unimodular()
LK3.signature_pair()

In [ ]:
LK3

## Free algebras on sets

The free commutative algebra on $\Delta[2]$ should look like a polynomial ring $R[x,y,z]$. The next cells keep the construction at the set level, then inspect the resulting polynomial behavior.

In [ ]:
A = Algebras(ZZ).Free().on(Sets.Δ[2])
a0, a1, a2 = A.generators()
(A, a0, a1, a2)

In [ ]:
P.<x, y, z> = PolynomialRing(ZZ, 3)
evaluation = A.Hom(P)(
    lambda monomial: prod(
        (P.gen(i)^e for i, e in monomial.dict().items()),
        P.one(),
    )
)
p = (a0 + a1)^2 * a2
(evaluation(p), (x + y)^2 * z)

A map of generating sets becomes substitution of variables. For $f\colon \Delta[2] \to \Delta[1]$ given by $i \mapsto i \bmod 2$, the induced map sends $a_0,a_1,a_2$ to $b_0,b_1,b_0$.

In [ ]:
S = Sets.Δ[2]
T = Sets.Δ[1]
B = Algebras(ZZ).Free().on(T)
f = SetMorphism(Hom(S, T, Sets()), lambda i: i % 2)
phi = A.induced_hom(f, B)
b0, b1 = B.generators()
(phi(a0), phi(a1), phi(a2), phi((a0 + a1) * a2))

The set map need not be finite. On $\mathbb{N}$, doubling acts on each finite expression without enumerating the whole generating set.

In [ ]:
U = Set(NN)
C = Algebras(ZZ).Free().on(U)
doubling = SetMorphism(Hom(U, U, Sets()), lambda n: 2 * n)
psi = C.induced_hom(doubling, C)
c3, c8 = C.algebra_generator(3), C.algebra_generator(8)
q = 2 * c3 + c8 * c3
(q, psi(q), 2 * C.algebra_generator(6) + C.algebra_generator(16) * C.algebra_generator(6))